# Delivery Time Prediction — Week 4

This notebook creates simulated logistics data, trains three models, and evaluates prediction quality. The data is hypothetical.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
rng = np.random.default_rng(42)
n = 2400
df = pd.DataFrame({
    "distance_km": rng.uniform(2, 180, n),
    "traffic_level": rng.integers(1, 6, n),
    "weather_severity": rng.integers(0, 4, n),
    "dispatch_delay_min": rng.gamma(2.0, 8.0, n),
    "package_weight_kg": rng.uniform(0.2, 35, n),
    "number_of_stops": rng.integers(1, 16, n),
    "warehouse_processing_min": rng.gamma(2.2, 7.0, n),
})
noise = rng.normal(0, 0.35, n)
df["delivery_time_hours"] = (0.12 + 0.018*df.distance_km + 0.16*df.traffic_level + 0.22*df.weather_severity + 0.012*df.dispatch_delay_min + 0.018*df.number_of_stops + 0.010*df.warehouse_processing_min + 0.002*df.package_weight_kg + noise).clip(lower=0.15)
df.head()

In [ ]:
print(df.describe().round(2))
print("\nMissing values:\n", df.isna().sum())
print("\nTarget correlations:\n", df.corr(numeric_only=True)["delivery_time_hours"].sort_values(ascending=False))

In [ ]:
target = "delivery_time_hours"
X, y = df.drop(columns=[target]), df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
models = {
    "Median baseline": DummyRegressor(strategy="median"),
    "Linear Regression": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler()), ("model", LinearRegression())]),
    "Random Forest": Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", RandomForestRegressor(n_estimators=250, max_depth=12, min_samples_leaf=3, random_state=42, n_jobs=-1))]),
}
rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append({"model": name, "MAE_hours": mean_absolute_error(y_test, pred), "RMSE_hours": mean_squared_error(y_test, pred)**0.5, "R2": r2_score(y_test, pred)})
results = pd.DataFrame(rows).sort_values("MAE_hours")
results.round(4)

## Interpretation and optimization

Compare MAE and RMSE; lower values are better. Review residuals by distance, traffic, weather, and stop count. In an operational setting, predictions could flag shipments at risk of missing promised windows. Validate with historical data and a time-based holdout before deployment.